In [1]:
import os
import sys
import tkinter
import abtem
print(os.getcwd())
path = "../styles/"
os.chdir(path)
print(os.getcwd())
import visualID as vID
from visualID import  fg, hl, bg


path = '../'
os.chdir(path)
print(os.getcwd())

import gzip
import numpy as np
from pyNanoMatBuilder import TEM_creator as tc
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data

import matplotlib.pyplot as plt
import numpy as np
import importlib
import ase 
from ase.io import read

C:\Users\saram\pynanomat_test\ML_HRTEM
C:\Users\saram\pynanomat_test\styles
C:\Users\saram\pynanomat_test


# HRTEM images database generation

This notebook demonstrates how to generate a database of HRTEM images for **Wulff** and **icosahedral** nanoparticles deposited on an amorphous carbon substrate (currently **5×5 nm**; **10×10 nm** support is in progress).

The workflow relies on two classes:

- `CreateHRTEMStructure`: generates `.xyz` files containing atomic coordinates for nanoparticles placed on the carbon substrate.
- `CreateHRTEMImage`: generates HRTEM images from those `.xyz` structures using **abTEM**.

For more details on imaging parameters, see the `HRTEM_params.ipynb` notebook and the abTEM documentation: <https://abtem.readthedocs.io/en/latest/intro.html>.

## File naming convention

### 1) XYZ files (`CreateHRTEMStructure`)
`compound_lattice_shape_0000001.xyz`

- The index encodes structure-generation settings (size/orientation combinations).
- Full structure metadata is saved in `xyz_metadata.csv` with the following fields:
  - **ID**: structure identifier (encodes size and orientation combination)
  - **orientation**: placement type (surface, edge, or vertex)
  - **angle_xy**: rotation angle in the xy-plane (carbon substrate plane)
  - **angle_tilt**: tilt angle along the z-axis (perpendicular to substrate)
  - **diameter**: circumsphere diameter (maximum nanoparticle size in nm)

### 2) PNG files (`CreateHRTEMImage`)
`compound_lattice_shape_0000001_0000000.png`

- First index: structure identifier (size/orientation combination).
- Second index: microscope/imaging parameter combination.
- Complete metadata is collected in the final dataframe generated at the end of this notebook.

## Why CSV output?

Metadata is exported to CSV to keep the dataset easy to inspect and use outside Python (for example, by experimental users).

---

## Important notes

### Size parameter interpretation

`size` is defined as a set of **multipliers** of the minimum size increment (typically \(d_{hkl}\)).

Examples:

- `size = np.arange(2, 150, 1)`  
  → sizes from **2×d(hkl)** to **149×d(hkl)** in **1×d(hkl)** steps.
- `size = np.arange(2, 150, 2)`  
  → sizes from **2×d(hkl)** to **148×d(hkl)** in **2×d(hkl)** steps.

### Orientation handling

Nanoparticles may be generated resting on surfaces, edges, or vertices (depending on geometry and current implementation state).  
Use `noOutput=False` to print orientation diagnostics:

- <span style="color:#008000"><strong>Green</strong>: edge configuration</span>  
- <span style="color:#0000FF"><strong>Blue</strong>: surface configuration</span>

### Structure validation

pyNanoMatBuilder ensures structural consistency by validating lattice-shape compatibility. Set `noOutput=False` for detailed reporting:
- <span style="color:#008000">**Green**: Successfully created structures (lattice-shape match)</span>
- <span style="color:#FF0000">**Red**: Rejected structures (lattice-shape mismatch)</span>

### Current limitations

- Some “edge” placements may effectively correspond to vertex contacts (ongoing development).
- Only Wulff and icosahedral nanoparticle families are currently supported.
- The NPs are always at the same place in the image; you can rotate them yourself using libraries like `scikit-image` or `PIL` for data augmentation in machine learning applications.



### 1. Generate the nanoparticles on the carbon substrate with different orientations (XYZ files)

Use the `CreateHRTEMStructure` class to generate XYZ files containing nanoparticles deposited on a carbon substrate with various orientations (surface, edge, or vertex contact). Currently supports Wulff and icosahedral nanoparticle geometries.

#### Input Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| **path** | str | Output directory for generated files |
| **xyz_gz_file** | str | Path to gzipped XYZ file containing the carbon substrate atomic coordinates |
| **cif_data** | DataFrame | Compound data from CIF files (pre-defined, extensible) |
| **wulff_shapes** | DataFrame | Wulff shape definitions and properties |
| **nRot** | int | Number of rotations for surface-oriented nanoparticles around the z-axis (rotation angle = 360°/nRot) |
| **sizes** | array | Size multipliers of the lattice spacing (d<sub>hkl</sub>) |
| **min_size** | float, optional | Minimum nanoparticle diameter (circumsphere). Default: 0 nm |
| **max_size** | float, optional | Maximum nanoparticle diameter (circumsphere). Default: 50 nm |
| **tolerance** | float, optional | Distance tolerance between nanoparticle and carbon substrate (Å). Use caution with small values to avoid spurious interfacial bonds |
| **noOutput** | bool | Suppress console output if `True` |

#### Output Files

- **XYZ files**: `compound_lattice_shape_0000001.xyz`  
  Atomic coordinates of nanoparticle-substrate structures. Index increments with size and orientation variations.

- **CSV metadata**: `xyz_metadata.csv`  
  Comprehensive structure information with columns:
  - **filename**: XYZ file name
  - **ID**: Structure identifier (encodes size and orientation)
  - **orientation**: Contact type (surface, edge, or vertex)
  - **angle_xy**: In-plane rotation angle (°, substrate plane)
  - **angle_tilt**: Out-of-plane tilt angle (°, perpendicular to substrate)
  - **diameter**: Nanoparticle circumsphere diameter (nm)



###  Example: 5x5 nm carbon substrate

In [2]:
t = pyNMBu.timer()
t.chrono_start()

# Paths 
# xyz_gz_file = 'cif_database/amorphousC/aC_relax_10x10.xyz.gz' # 1Ox1O nm, the NP is placed on the corner of the substrate in this case (not so good?)
xyz_gz_file = 'cif_database/amorphousC/aC_relax_5x5.xyz.gz' # 5x5 nm

path='ML_HRTEM/HRTEM_xyz' 

# Sizes of the NPs
# 1. Size increment
size = np.arange(2, 50, 1) # d is a multiplier coefficient of dhkl, so the final sizes are : 2*dhkl, 3*dhkl, ..., 10*dhkl
# 2. Min size (nm)
min_size = 1
# 3. Max size (nm)
max_size = 1.5

# Angles (rotation of the NP along z)
nRot=2

# Instance of the class
TEM_class_test=tc.CreateHRTEMStructure(path ,xyz_gz_file,                    # paths
                                       cif_data=data.pyNMBcif.CIFdf,         # compounds
                                       wulff_shapes=data.WulffShapes.WSdf,   # all wulff shapes
                                       nRot= nRot,                           # number of orientations
                                       sizes=size,min_size = min_size, max_size = max_size, # size informations (increment, min, max)
                                       tolerance=3,                          # distance tolerance between the NP and the carbon grid 
                                       noOutput=False)

print(t.chrono_stop(hdelay=True))


                         NaCl                       

Absolute path to CIF: C:\Users\saram\pynanomat_test\cif_database\cod1000041-NaCl.cif
 d_hkl=0.562 nm 

                     TiO2 rutile                    

Absolute path to CIF: C:\Users\saram\pynanomat_test\cif_database\cod9015662-TiO2-rutile.cif
 d_hkl=0.29587 nm 
 No interatomic distance found.

                     TiO2 anatase                   

Absolute path to CIF: C:\Users\saram\pynanomat_test\cif_database\cod9015929-TiO2-anatase.cif
 d_hkl=0.9514300000000001 nm 
 No interatomic distance found.

                        Fe bcc                      

Absolute path to CIF: C:\Users\saram\pynanomat_test\cif_database\cod5000217-Fe_bcc.cif
 d_hkl=0.28608 nm 

  bcc corresponds to the lattices ['bcc', 'fcc'] of the Wulff cube form.  



C:\Users\saram\anaconda3\envs\abtem_env\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(
C:\Users\saram\pynanomat_test\pyNanoMatBuilder\utils.py:364: FutureWarning: `get_spacegroup` has been deprecated due to its misleading output. The returned `Spacegroup` object has symmetry operations for a standard setting regardress of the given `Atoms` object. See https://gitlab.com/ase/ase/-/issues/1534 for details. Please use `ase.spacegroup.symmetrize.check_symmetry` or `spglib` directly to get the symmetry operations for the given `Atoms` object.
  self.ucSG = get_spacegroup(self.cif, symprec=float(1e-2))
C:\Users\saram\pynanomat_test\pyNanoMatBuilder\utils.py:364: FutureWarning: `get_spacegroup` has been deprecated due to its misleading output. The returned `Spacegroup` object has symmetry operations for a standard setting regardress of the given `Atoms` o

 The circumscribed sphere diameter of the NP =0.9910101900586089 nm is smaller than the minimal size : 1nm chosen. 
 Generating size is 0.8582 nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =1.4865152850879124 nm and is in the interval [1,1.5].

 Generate NPs laying on one of their surfaces.
Surface plane of the NP used = [-0.     -0.     -1.     -4.2912].
  File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_cube_000001.xyz 

  File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_cube_000002.xyz 

 Generate NPs laying on one of their edges.
Two adjacent planes found.
Angle entre les 2 plans = 90.00°
Angle max entre les plans et le plan (xy) = 45.00°
 File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_cube_000003.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_cube_000004.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_cube_000005.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_cube_000006.xyz 



C:\Users\saram\anaconda3\envs\abtem_env\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(
C:\Users\saram\pynanomat_test\pyNanoMatBuilder\utils.py:99: FutureWarning: `get_spacegroup` has been deprecated due to its misleading output. The returned `Spacegroup` object has symmetry operations for a standard setting regardress of the given `Atoms` object. See https://gitlab.com/ase/ase/-/issues/1534 for details. Please use `ase.spacegroup.symmetrize.check_symmetry` or `spglib` directly to get the symmetry operations for the given `Atoms` object.
  system.ucSG = get_spacegroup(system.cif, symprec=system.aseSymPrec)


 The circumscribed sphere diameter of the NP =1.9820203801172178 nm is greater than the maximal size : 1.5nm chosen. 

  bcc corresponds to the lattices ['bcc'] of the Wulff bccrDD form.  

 The circumscribed sphere diameter of the NP =0.57216 nm is smaller than the minimal size : 1nm chosen. 
 Generating size is 0.8582 nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =1.1443200000000002 nm and is in the interval [1,1.5].

 Generate NPs laying on one of their surfaces.
Surface plane of the NP used = [-1.64648957e-16 -7.07106781e-01  7.07106781e-01 -4.04578216e+00].
  File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_bccrDD_000007.xyz 

  File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_bccrDD_000008.xyz 

 Generate NPs laying on one of their edges.
Two adjacent planes found.
Angle entre les 2 plans = 60.00°
Angle max entre les plans et le plan (xy) = 60.00°
 File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_bccrDD_000009.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Fe_bcc_bccrDD_000010.xy

...

C:\Users\saram\pynanomat_test\pyNanoMatBuilder\utils.py:99: FutureWarning: `get_spacegroup` has been deprecated due to its misleading output. The returned `Spacegroup` object has symmetry operations for a standard setting regardress of the given `Atoms` object. See https://gitlab.com/ase/ase/-/issues/1534 for details. Please use `ase.spacegroup.symmetrize.check_symmetry` or `spglib` directly to get the symmetry operations for the given `Atoms` object.
  system.ucSG = get_spacegroup(system.cif, symprec=system.aseSymPrec)


 Generating size is 1.2259 nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =1.2921698974980036 nm and is in the interval [1,1.5].

 Generate NPs laying on one of their surfaces.
Surface plane of the NP used = [-0.57735027 -0.57735027 -0.57735027 -4.71833734].
  File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_dicoTd_000235.xyz 

  File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_dicoTd_000236.xyz 

 Generate NPs laying on one of their edges.
Two adjacent planes found.
Angle entre les 2 plans = 54.74°
Angle max entre les plans et le plan (xy) = 62.63°
 File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_dicoTd_000237.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_dicoTd_000238.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_dicoTd_000239.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_dicoTd_000240.xyz 



C:\Users\saram\pynanomat_test\pyNanoMatBuilder\utils.py:99: FutureWarning: `get_spacegroup` has been deprecated due to its misleading output. The returned `Spacegroup` object has symmetry operations for a standard setting regardress of the given `Atoms` object. See https://gitlab.com/ase/ase/-/issues/1534 for details. Please use `ase.spacegroup.symmetrize.check_symmetry` or `spglib` directly to get the symmetry operations for the given `Atoms` object.
  system.ucSG = get_spacegroup(system.cif, symprec=system.aseSymPrec)


 The circumscribed sphere diameter of the NP =1.7336278375706824 nm is greater than the maximal size : 1.5nm chosen. 
Addinng icosahedron with crystal type fcc and interatomic distance = 2.8894 nm.
 Number of bonds is 1
 The circumscribed sphere diameter of the NP =0.5495926839174244 nm is smaller than the minimal size : 1nm chosen. 
 Number of bonds is 2
 Generating size is 1.6345 nm. 
  Circumscribed sphere diameter =1.0991853678348489 nm and is in the interval [1,1.5].

 Generate NPs laying on one of their surfaces.
Surface plane of the NP used = [-0.57735027  0.57735027  0.57735027 -4.36736284].
  File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_regico_000241.xyz 

  File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_regico_000242.xyz 

 Generate NPs laying on one of their edges.
Two adjacent planes found.
Angle entre les 2 plans = 41.81°
Angle max entre les plans et le plan (xy) = 69.09°
 File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_regico_000243.xyz 

 File written : ML_HRTEM/HRTEM_xyz/Ag_fcc_regico_0

C:\Users\saram\pynanomat_test\pyNanoMatBuilder\utils.py:364: FutureWarning: `get_spacegroup` has been deprecated due to its misleading output. The returned `Spacegroup` object has symmetry operations for a standard setting regardress of the given `Atoms` object. See https://gitlab.com/ase/ase/-/issues/1534 for details. Please use `ase.spacegroup.symmetrize.check_symmetry` or `spglib` directly to get the symmetry operations for the given `Atoms` object.
  self.ucSG = get_spacegroup(self.cif, symprec=float(1e-2))


### 2. Use abTEM to create the HRTEM images




#### Generate HRTEM images with `CreateHRTEMImage` (from `TEM_creator`)

This section uses the `CreateHRTEMImage` class (built on **abTEM**) to generate HRTEM images from the XYZ structures created in the previous step.  
abTEM documentation: <https://abtem.readthedocs.io/en/latest/intro.html>

For a detailed explanation of imaging parameters, see *HRTEM_parameters.ipynb* (in progress).

Parameter ranges used here are guided by published work and internal benchmark tests, including:  
*“Simulated HRTEM images of nanoparticles to train a neural network for crystallinity classification”*  
<https://www.sciencedirect.com/science/article/pii/S0304399122001607>

In the example below, selected microscope parameters are scanned in a loop to generate multiple image conditions.  
This is a template workflow—adjust parameter lists according to your experimental setup or modeling goals.

> **Note:** This parameter set is continuously refined. Feedback on missing or better-calibrated parameters is welcome.

In [3]:
tc.CreateHRTEMImage?

Init signature:
tc.CreateHRTEMImage(
    path_input: str,
    path_output: str,
    sampling: float = 0.05,
    masking_images: bool = True,
    phonon_config: int = 8,
    Cs_value: float = -80000.0,
    C12: float = 0,
    phi12: float = 0,
    Cc_value: float = 10000000.0,
    semiangle_cutoff_value: int = 45,
    energy_spread: float = 0.35,
    c1: float = -0.6,
    c2: float = 0.1,
    c3: float = 1.0,
    dose_poisson_noise: float = 10000.0,
    sigmas: float = 0.1,
    slice_thickness: float = 1,
    noOutput: bool = True,
    energy: float = 200000.0,
    substrate_size: list = [10, 10, 10],
    device: str = 'gpu',
)
Docstring:     
Simulate High-Resolution Transmission Electron Microscopy (HRTEM) images
from XYZ atomic structure files using the abTEM multislice framework.

This class reads XYZ files produced by ``CreateHRTEMStructure`` (nanoparticle
on an amorphous-carbon substrate) and generates realistic HRTEM images by
chaining the following physical steps:

**Simulation 

In [ ]:
import numpy as np
from itertools import product
import glob
import os


# Directory with only one structure (XYZ file)
path_input = 'ML_HRTEM/HRTEM_xyz'
 # Directory of the output HRTEM images (PNG format)
path_output = 'ML_HRTEM/HRTEM_png'  

# Define the parameters you want to play with and make lists
all_semiangle_cutoff_value=[30] # 40
all_phonon_config_value = [8]
all_cs_value = [-1.5e4] # Angs
all_Cc_value = [1e-3 * 1e10] # Angs
all_substrate_size = [[5,5,5]] # substrate total size nm
all_astigmatism_value = [0]
all_c1 = [0]
all_c2 = [0.5]
all_c3 = [2]

# Generate all combinations of parameters
parameter_combinations = list(product( all_semiangle_cutoff_value,all_phonon_config_value,all_cs_value,all_astigmatism_value, all_Cc_value, all_substrate_size, all_c1, all_c2, all_c3))
# Calculate total combinations
total_combinations = len(parameter_combinations)
print('total_combinations',total_combinations)


# Initialize a counter for the current step
current_step = 0

# Loop through all combinations of parameters
for semiangle_cutoff_value,phonon_config_value, Cs_value,astigmatism_value, Cc_value, substrate_size, c1, c2, c3 in parameter_combinations:
    
    current_step += 1
    print(f"Step {current_step} on {total_combinations}")

    # Create the HRTEM images, some parameters are set in this example
    microscope_image = tc.CreateHRTEMImage(
        path_input = path_input,
        path_output = path_output,
        sampling = 0.1,
        masking_images = True,
        phonon_config= phonon_config_value,
        Cs_value=Cs_value, 
        Cc_value= Cc_value,
        C12 = 0,
        phi12 = 0, 
        semiangle_cutoff_value=semiangle_cutoff_value,
        energy_spread=0.35,
        c1 = c1,
        c2 = c2,
        c3 = c3,
        dose_poisson_noise = 1e4,
        sigmas = 0.1,
        slice_thickness= 1,
        noOutput=False,
        energy=200e3,        
        substrate_size=substrate_size,
        device='cpu'
    )
    

total_combinations 1
Step 1 on 1
[########################################] | 100% Completed | 4.53 ss
defocus = -23.75 Å
focal_spread = 17.50 Å  (Cc=1.00e+07 Å, dE=0.35 eV)
File is ML_HRTEM/HRTEM_png/Ag_fcc_cube_000205_0000001.png
[CSV] Saved metadata: ML_HRTEM/HRTEM_png/Ag_fcc_cube_000205_0000001_metadata.csv
--------Finished---------------
Mask image saved as ML_HRTEM/HRTEM_png/Ag_fcc_cube_000205_0000001_mask.png
[########################################] | 100% Completed | 1.89 ss
defocus = -23.75 Å
focal_spread = 17.50 Å  (Cc=1.00e+07 Å, dE=0.35 eV)
File is ML_HRTEM/HRTEM_png/Ag_fcc_cube_000206_0000002.png
[CSV] Saved metadata: ML_HRTEM/HRTEM_png/Ag_fcc_cube_000206_0000002_metadata.csv
--------Finished---------------
Mask image saved as ML_HRTEM/HRTEM_png/Ag_fcc_cube_000206_0000002_mask.png
[########################################] | 100% Completed | 2.11 ss
defocus = -23.75 Å
focal_spread = 17.50 Å  (Cc=1.00e+07 Å, dE=0.35 eV)
File is ML_HRTEM/HRTEM_png/Ag_fcc_cube_000207_000000

*NB: if you use GPU, make sure to install Cupy (https://cupy.dev/) first !*

#### Function to generate CSV metadata for HRTEM images

The `tc.create_csv()` function extracts and consolidates metadata from all generated HRTEM images (PNG files) into a single CSV file for easy inspection and downstream analysis.

**Input parameters:**
- `tem_image_paths`: Directory containing PNG images
- `output_csv`: Output CSV filename
- `noOutput`: Suppress console output if `True`

**Output:**
- CSV file with complete imaging parameters and structure identifiers for each generated image

In [4]:
tem_image_paths = 'ML_HRTEM/HRTEM_png'
output_csv = 'metadata_images.csv'

tc.create_csv(tem_image_paths, output_csv, noOutput = True)

,id,element,crystal_structure,shape,orientation,angle_xy_deg,angle_tilt_deg,circumsphere_diameter_nm,sampling_A-1,phonon_config,...,semiangle_cutoff_value_mrad,energy_eV,mtf_c1,mtf_c2,mtf_c3,dose_poisson_noise_e-A-2,sigmas_A,slice_thickness_A,substrate_size_x_nm,substrate_size_y_nm
0,Ag_fcc_cube_000205_0000001,Ag,fcc,cube,surface,139.0,0.0,1.4155,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
1,Ag_fcc_cube_000206_0000002,Ag,fcc,cube,surface,123.0,0.0,1.4155,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
2,Ag_fcc_cube_000207_0000003,Ag,fcc,cube,edge,116.0,-35.0,1.4155,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
3,Ag_fcc_cube_000208_0000004,Ag,fcc,cube,edge,96.0,-35.0,1.4155,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
4,Ag_fcc_cube_000209_0000005,Ag,fcc,cube,edge,94.0,-17.0,1.4155,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,Ru_hcp_hcpsph1_000117_0000248,Ru,hcp,hcpsph1,edge,60.0,-40.0,1.0464,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
256,Ru_hcp_hcpsph1_000118_0000249,Ru,hcp,hcpsph1,edge,127.0,-40.0,1.0464,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
257,Ru_hcp_hcpsph1_000119_0000250,Ru,hcp,hcpsph1,edge,95.0,15.0,1.0464,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
258,Ru_hcp_hcpsph1_000120_0000251,Ru,hcp,hcpsph1,edge,146.0,15.0,1.0464,0.1,8,...,30,200000.0,0,0.5,2,10000.0,0.1,1,5,5
